# GUS04D — Variable-Specific Estimation Pipelines

Run demographic estimation for all variable types using the three-layer pipeline:
- **Layer 1**: Log-linear interpolation (seed generation)
- **Layer 2**: Marginal fitting via IPF (not used in age×sex_2000)
- **Layer 3**: Hierarchical consistency (residual scaling / Gurobi QP)

**Prerequisites**: Run GUS02B → GUS03 to produce `geoteryt_O.pkl`.

In [4]:
# ── Cell 1: Imports & load database ──
import sys, os, time
import numpy as np
import pandas as pd

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'Code', 'tools'))
DATA_ROOT = os.path.join(REPO, '..', '..', 'Data', 'Geospatial')

from geoTERYT_db import (
    load_complete_database, LEVEL_GMINA, LEVEL_VOIVODESHIP, LEVEL_POWIAT,
)

db_path = os.path.join(DATA_ROOT, 'geoteryt_O.pkl')
print(f"Loading database from {db_path}…")
t0 = time.time()
db = load_complete_database(db_path)
print(f"Loaded in {time.time()-t0:.1f}s — {len(db._records)} records")

Loading database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl…
Loading complete database from /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/local_repo/LRDWI-Paper/../../Data/Geospatial/geoteryt_O.pkl...
  Database version: 4.3
  ✓ Restored old voivodships: 49 rows
  ✓ Restored geometry store: 13,287 unique geometries
  ✓ Restored geometry data for years: [2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2015, 2016, 2017, 2018, 2021, 2022, 2023]
  ✓ Loaded 4612 records
  ✓ Year range: 1999 - 2024
  ✓ Records with geometry: 3661
  ✓ Records with old_woj: 4104
  ✓ Records with data: 4584
  ✓ Records with cross tables: 4584
  ✓ Records with population data: 4582
  ✓ Records with pop_class: 3411
Loaded in 385.3s — 4612 records


In [11]:
# ── Cell 2: Initialize DemographicEstimator ──
import importlib
import demographic_estimator
importlib.reload(demographic_estimator)
from demographic_estimator import DemographicEstimator, PREDICTION_2000_RANGE, PREDICTION_1990_RANGE

est = DemographicEstimator(db, verbose=True)
print(repr(est))

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)
DemographicEstimator(completed=0/9, Gurobi=YES)


## Age × Sex — Prediction2000 (1999–2025)

Sources: M_pop__age_sex (BDL P2137 + Census 2002/2011/2021), shape (19, 3).  
Coverage: ~87–95% of gminas per year. Missing gminas filled via log-linear interpolation + voivodeship residual scaling.

In [12]:
# ── Cell 3: Run age×sex Prediction2000 ──
t0 = time.time()
est.run_pipeline('age_sex', '2000')
print(f"\nCompleted in {time.time()-t0:.1f}s")


  PIPELINE: age_sex / Prediction2000
  Output subject: E_age_sex_2000
  Source: M_pop__age_sex  shape=(19, 3)
  Gminas total: 2671, with M_pop__age_sex: 2618 (skipped 53 with mismatched shape)
  Layer 1: generating seeds (log-linear interpolation)…
    Seeds generated: 2618/2618 units (skipped 0)
    2000: 2442 obs + 0 est
    2005: 2478 obs + 0 est
    2010: 2479 obs + 0 est
    2015: 2478 obs + 0 est
    2020: 2477 obs + 0 est
    2025: 0 obs + 2477 est
  Aggregating to powiat and voivodeship levels…
    Aggregated: 10226 powiat-years, 432 voiv-years
  Summary: 64318 observed + 2477 estimated cell-years stored
  ✓  E_age_sex_2000 complete

Completed in 17.5s


In [13]:
# ── Cell 4: Validation — coverage and provenance ──
e_sid = 'E_age_sex_2000'

# Count records with E_ data by level
by_level = {'gmina': 0, 'powiat': 0, 'voiv': 0}
by_level_total = {'gmina': 0, 'powiat': 0, 'voiv': 0}

for tid, rec in db._records.items():
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}:
        by_level_total['gmina'] += 1
        if e_sid in rec.cross_tables:
            ct = rec.cross_tables[e_sid]
            if ct.years_with_data:
                by_level['gmina'] += 1
    elif rec.level == LEVEL_POWIAT:
        by_level_total['powiat'] += 1
        if e_sid in rec.cross_tables:
            ct = rec.cross_tables[e_sid]
            if ct.years_with_data:
                by_level['powiat'] += 1
    elif rec.level == LEVEL_VOIVODESHIP:
        by_level_total['voiv'] += 1
        if e_sid in rec.cross_tables:
            ct = rec.cross_tables[e_sid]
            if ct.years_with_data:
                by_level['voiv'] += 1

print(f"E_age_sex_2000 coverage:")
for lvl in ['gmina', 'powiat', 'voiv']:
    print(f"  {lvl}: {by_level[lvl]}/{by_level_total[lvl]}")

# Provenance summary
prov_df = est.get_provenance_summary(e_sid)
if not prov_df.empty:
    print(f"\nProvenance (by year):")
    print(prov_df.to_string())

E_age_sex_2000 coverage:
  gmina: 2618/2671
  powiat: 381/382
  voiv: 16/67

Provenance (by year):
      n_units  mean_frac_observed  min_frac_observed
year                                                
1999     3015            0.808292                0.0
2000     3015            0.809950                0.0
2001     3015            0.811277                0.0
2002     3015            0.821891                0.0
2003     3015            0.821891                0.0
2004     3015            0.821891                0.0
2005     3015            0.821891                0.0
2006     3015            0.821891                0.0
2007     3015            0.821891                0.0
2008     3015            0.821891                0.0
2009     3015            0.821891                0.0
2010     3015            0.822222                0.0
2011     3015            0.822222                0.0
2012     3015            0.822222                0.0
2013     3015            0.822222                0.0


In [14]:
# ── Cell 5: Validation — voivodeship consistency ──
# Compare estimated voivodeship E_ totals with observed M_pop__age_sex

source_sid = 'M_pop__age_sex'
e_sid = 'E_age_sex_2000'
voiv_tids = sorted(tid for tid in db._by_level.get(LEVEL_VOIVODESHIP, set()))

errors = []
for voiv_tid in voiv_tids:
    rec = db._records.get(voiv_tid)
    if rec is None:
        continue
    obs_ct = rec.cross_tables.get(source_sid)
    est_ct = rec.cross_tables.get(e_sid)
    if obs_ct is None or est_ct is None:
        continue
    
    for year in PREDICTION_2000_RANGE:
        obs_tbl = obs_ct.tables.get(year)
        est_tbl = est_ct.tables.get(year)
        if obs_tbl is None or np.all(np.isnan(obs_tbl)):
            continue
        if est_tbl is None or np.all(np.isnan(est_tbl)):
            continue
        # Compare grand totals (ogółem×ogółem)
        obs_total = np.nansum(obs_tbl)  # approximate: includes ogółem
        est_total = np.nansum(est_tbl)
        
        # Better: compare core cells only
        # Find ogółem index
        dim_names = obs_ct.dim_names
        dim_labels = obs_ct.dim_labels
        og0 = next((i for i, l in enumerate(dim_labels[dim_names[0]]) if l.lower()=='ogółem'), None)
        og1 = next((i for i, l in enumerate(dim_labels[dim_names[1]]) if l.lower()=='ogółem'), None)
        
        if og0 is not None and og1 is not None:
            obs_grand = obs_tbl[og0, og1]
            est_grand = est_tbl[og0, og1]
            if obs_grand > 0:
                pct_err = 100 * (est_grand - obs_grand) / obs_grand
                errors.append({
                    'voiv': voiv_tid, 'name': rec.name, 'year': year,
                    'obs_pop': obs_grand, 'est_pop': est_grand,
                    'pct_error': pct_err
                })

if errors:
    err_df = pd.DataFrame(errors)
    print(f"Voivodeship consistency: {len(err_df)} comparisons")
    print(f"  Mean abs % error: {err_df['pct_error'].abs().mean():.3f}%")
    print(f"  Max  abs % error: {err_df['pct_error'].abs().max():.3f}%")
    worst = err_df.loc[err_df['pct_error'].abs().idxmax()]
    print(f"  Worst: {worst['name']} ({worst['voiv']}) year {worst['year']}: "
          f"{worst['pct_error']:.3f}%")
else:
    print("No voivodeship comparisons available")

Voivodeship consistency: 416 comparisons
  Mean abs % error: 0.353%
  Max  abs % error: 33.841%
  Worst: MAZOWIECKIE (1400000) year 1999: -33.841%


In [15]:
# ── Cell 6: Spot-check — sample gmina time series ──
# Pick 3 gminas: one with full data, one with partial data, one estimated-heavy

sample_tids = []
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid)
    if ect is None:
        continue
    ywd = ect.years_with_data
    if len(sample_tids) == 0 and len(ywd) >= 25:
        sample_tids.append(tid)  # well-covered
    elif len(sample_tids) == 1 and 10 <= len(ywd) < 25:
        sample_tids.append(tid)  # partial
    elif len(sample_tids) == 2:
        break
# If we didn't find a partial one, just use a different well-covered one
if len(sample_tids) < 2:
    for tid, rec in db._records.items():
        if tid not in sample_tids and rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}:
            ect = rec.cross_tables.get(e_sid)
            if ect and ect.years_with_data:
                sample_tids.append(tid)
                break

for tid in sample_tids[:3]:
    rec = db._records[tid]
    ect = rec.cross_tables[e_sid]
    print(f"\n{'='*60}")
    print(f"{rec.name} ({tid}) — {len(ect.years_with_data)} years with E_ data")
    
    # Show ogółem time series (grand total = population check)
    dim_names = ect.dim_names
    dim_labels = ect.dim_labels
    og0 = next((i for i, l in enumerate(dim_labels[dim_names[0]]) if l.lower()=='ogółem'), None)
    og1 = next((i for i, l in enumerate(dim_labels[dim_names[1]]) if l.lower()=='ogółem'), None)
    
    print(f"Year | E_total  | pop      | source_M | match?")
    print(f"-----|----------|----------|----------|-------")
    source_ct = rec.cross_tables.get(source_sid)
    for year in PREDICTION_2000_RANGE:
        tbl = ect.tables.get(year)
        if tbl is None or np.all(np.isnan(tbl)):
            continue
        e_total = tbl[og0, og1] if og0 is not None and og1 is not None else np.nansum(tbl)
        
        pop_val = rec.pop.get(pd.Timestamp(year, 1, 1), np.nan)
        
        m_total = np.nan
        if source_ct:
            m_tbl = source_ct.tables.get(year)
            if m_tbl is not None and not np.all(np.isnan(m_tbl)):
                m_total = m_tbl[og0, og1] if og0 is not None and og1 is not None else np.nansum(m_tbl)
        
        is_obs = "OBS" if not np.isnan(m_total) else "EST"
        match = "✓" if abs(e_total - pop_val) < 1 else f"Δ={e_total-pop_val:.1f}" if not np.isnan(pop_val) else "?"
        print(f"{year} | {e_total:>8.0f} | {pop_val:>8.0f} | {m_total:>8.0f} | {is_obs} {match}")


Bolesławiec (0201011) — 27 years with E_ data
Year | E_total  | pop      | source_M | match?
-----|----------|----------|----------|-------
1999 |    41914 |    41914 |    41914 | OBS ✓
2000 |    41731 |    41731 |    41731 | OBS ✓
2001 |    41646 |    41646 |    41646 | OBS ✓
2002 |    41371 |    41371 |    41371 | OBS ✓
2003 |    41263 |    41263 |    41263 | OBS ✓
2004 |    41117 |    41117 |    41117 | OBS ✓
2005 |    40984 |    40984 |    40984 | OBS ✓
2006 |    40679 |    40679 |    40679 | OBS ✓
2007 |    40384 |    40384 |    40384 | OBS ✓
2008 |    40258 |    40258 |    40258 | OBS ✓
2009 |    40021 |    40021 |    40021 | OBS ✓
2010 |    40309 |    40309 |    40309 | OBS ✓
2011 |    40119 |    40119 |    40119 | OBS ✓
2012 |    39851 |    39851 |    39851 | OBS ✓
2013 |    39603 |    39603 |    39603 | OBS ✓
2014 |    39464 |    39464 |    39464 | OBS ✓
2015 |    39373 |    39373 |    39373 | OBS ✓
2016 |    39167 |    39167 |    39167 | OBS ✓
2017 |    39084 |    39084 |   

## Age × Sex — Prediction1990 (1986–2002)

Sources: M_age_sex (16×3): P2137 (BDL gmina 1995–2024) + H_age_sex (old voivodeships 1986–1994).  
Census 1988: P2884 (age 10yr bins, gmina) + P2883 (sex, gmina).  
Challenge: 1988 census gives age/sex separately → must construct 2D joint table via IPF.

In [20]:
# ── Cell 7: Run age×sex Prediction1990 ──
import importlib, demographic_estimator
importlib.reload(demographic_estimator)
from demographic_estimator import DemographicEstimator

est = DemographicEstimator(db, verbose=True)
est._completed.add(('age_sex', '2000'))

t0 = time.time()
est.run_pipeline('age_sex', '1990')
print(f"\nCompleted in {time.time()-t0:.1f}s")

DemographicEstimator initialised  (Gurobi=YES, IPFN=YES)

  PIPELINE: age_sex / Prediction1990
  Output subject: E_age_sex_1990
  Source: M_age_sex  shape=(16, 3)
  Phase A: constructing 1988 gmina age×sex via IPF…
    1988 IPF: 2478 OK, 193 skipped
  Phase B: building seeds (log-linear interpolation)…
    Seeds: 2671 gminas
  Phase C: old voivodeship marginal scaling (1986–1994)…
    Scaled 441 old-voi × year combinations
  Phase D: storing results…
  Summary: 45407 cell-years for 2671 gminas
  ✓  E_age_sex_1990 complete

Completed in 21.0s


In [21]:
# ── Cell 8: Validation — E_age_sex_1990 quality checks ──
e_sid_1990 = 'E_age_sex_1990'
source_sid_1990 = 'M_age_sex'

# 1. Coverage
n_gminas_with = sum(
    1 for tid, rec in db._records.items()
    if rec.level == LEVEL_GMINA and tid[-1] in {'1','2','3'}
    and e_sid_1990 in rec.cross_tables
    and rec.cross_tables[e_sid_1990].years_with_data
)
print(f"E_age_sex_1990 coverage: {n_gminas_with} gminas")

# 2. Census 1988 vs population consistency
errors_1988 = []
for tid, rec in db._records.items():
    if rec.level != LEVEL_GMINA or tid[-1] not in {'1','2','3'}:
        continue
    ect = rec.cross_tables.get(e_sid_1990)
    if ect is None:
        continue
    tbl = ect.tables.get(1988)
    if tbl is None or np.all(np.isnan(tbl)):
        continue
    
    # Find ogółem×ogółem (grand total)
    og0 = next((i for i, l in enumerate(ect.dim_labels[ect.dim_names[0]]) if l.lower()=='ogółem'), None)
    og1 = next((i for i, l in enumerate(ect.dim_labels[ect.dim_names[1]]) if l.lower()=='ogółem'), None)
    if og0 is None or og1 is None:
        continue
    e_total = tbl[og0, og1]
    pop88 = rec.pop.get(pd.Timestamp(1988,1,1), np.nan)
    if not np.isnan(pop88) and pop88 > 0:
        pct_err = 100 * (e_total - pop88) / pop88
        errors_1988.append({'tid': tid, 'name': rec.name, 'e_total': e_total, 'pop': pop88, 'pct_err': pct_err})

err_df = pd.DataFrame(errors_1988)
if not err_df.empty:
    print(f"\n1988 E_ vs pop consistency ({len(err_df)} gminas):")
    print(f"  Mean abs % error: {err_df['pct_err'].abs().mean():.4f}%")
    print(f"  Max  abs % error: {err_df['pct_err'].abs().max():.4f}%")
    print(f"  Within 1%: {(err_df['pct_err'].abs() < 1).sum()}/{len(err_df)}")
    print(f"  Within 5%: {(err_df['pct_err'].abs() < 5).sum()}/{len(err_df)}")

# 3. Old voivodeship consistency for 1988
print(f"\n1988 old-voivodeship consistency:")
country_rec = db._records.get('0000000')
old_voi_tids = country_rec.children_ids.get('old', []) if country_rec else []
ov_errors = []
for ov_tid in old_voi_tids:
    ov_rec = db._records.get(ov_tid)
    if ov_rec is None:
        continue
    ov_ct = ov_rec.cross_tables.get(source_sid_1990)
    if ov_ct is None:
        continue
    ov_tbl = ov_ct.tables.get(1988)
    if ov_tbl is None or np.all(np.isnan(ov_tbl)):
        continue
    
    # Sum gmina E_ tables
    children = ov_rec.get_children(1988)
    agg = np.zeros_like(ov_tbl)
    n_children = 0
    for g in children:
        grec = db._records.get(g)
        if grec is None:
            continue
        gct = grec.cross_tables.get(e_sid_1990)
        if gct is None:
            continue
        gtbl = gct.tables.get(1988)
        if gtbl is None or np.all(np.isnan(gtbl)):
            continue
        if gtbl.shape != ov_tbl.shape:
            continue
        agg += np.nan_to_num(gtbl, nan=0.0)
        n_children += 1
    
    og0 = next((i for i, l in enumerate(ov_ct.dim_labels[ov_ct.dim_names[0]]) if l.lower()=='ogółem'), None)
    og1 = next((i for i, l in enumerate(ov_ct.dim_labels[ov_ct.dim_names[1]]) if l.lower()=='ogółem'), None)
    if og0 is not None and og1 is not None:
        ov_total = ov_tbl[og0, og1]
        agg_total = agg[og0, og1]
        if ov_total > 0:
            pct = 100 * (agg_total - ov_total) / ov_total
            ov_errors.append({'ov': ov_tid, 'name': ov_rec.name, 'pct_err': pct, 'n_children': n_children})

ov_df = pd.DataFrame(ov_errors)
if not ov_df.empty:
    print(f"  {len(ov_df)} old voivodeships compared")
    print(f"  Mean abs % error: {ov_df['pct_err'].abs().mean():.4f}%")
    print(f"  Max  abs % error: {ov_df['pct_err'].abs().max():.4f}%")
    worst = ov_df.loc[ov_df['pct_err'].abs().idxmax()]
    print(f"  Worst: {worst['name']} ({worst['ov']}): {worst['pct_err']:.2f}% ({int(worst['n_children'])} children)")

E_age_sex_1990 coverage: 2671 gminas

1988 E_ vs pop consistency (2490 gminas):
  Mean abs % error: 0.0000%
  Max  abs % error: 0.0000%
  Within 1%: 2490/2490
  Within 5%: 2490/2490

1988 old-voivodeship consistency:
  49 old voivodeships compared
  Mean abs % error: 5.8648%
  Max  abs % error: 62.3866%
  Worst: Warszawskie (5800000): -62.39% (51 children)


In [43]:
# Very concise Warsaw check
p = db.get_by_teryt_id('1431000')
for yr in [1999, 2000, 2001, 2002, 2003]:
    ch = p.get_children(yr) if p else []
    r123 = [g for g in ch if db.get_by_teryt_id(g) and db.get_by_teryt_id(g).rodz in ('1','2','3')]
    t = sum(db.get_by_teryt_id(g).pop.get(pd.Timestamp(yr,1,1), 0) for g in r123)
    pp = p.pop.get(pd.Timestamp(yr,1,1), np.nan) if p else np.nan
    ww = [g for g in r123 if g.startswith('1431')]
    print(f"yr={yr}: n_r123={len(r123):3d}  n_warsaw={len(ww):2d}  sum_pop={t:.0f}  pow_pop={pp:.0f}  ratio={t/pp:.3f}" if pp else f"yr={yr}: no pop")

yr=1999: n_r123= 12  n_warsaw=12  sum_pop=3354632  pow_pop=1677316  ratio=2.000
yr=2000: n_r123= 12  n_warsaw=12  sum_pop=3344836  pow_pop=1672418  ratio=2.000
yr=2001: n_r123= 12  n_warsaw=12  sum_pop=3343454  pow_pop=1671727  ratio=2.000
yr=2002: n_r123= 12  n_warsaw=12  sum_pop=nan  pow_pop=nan  ratio=nan
yr=2003: n_r123= 12  n_warsaw=12  sum_pop=nan  pow_pop=nan  ratio=nan
